# Modify CAM output


<div class="alert alert-info">
<strong>Exercise: Customize your CAM history files</strong><br><br>

Create a case called `b1850_high_freq` using the compset `B1850C_LTso` at resolution `ne16pg3_t201`. Use `--run-unsupported` because the 2 degree resolution is not supported at this stage.

We will interpolate CAM history output to a regular 2° × 2° latitude–longitude grid.


In addition to the monthly h0 output, tell CAM to output:

- Instantaneous values of  `T`, `Q`, `U`, and `V` every 24 hours.
- Time-averaged values of  `T`, `Q`, `U`, and `V` every 3 hours.

Set the namelist so that you produce:

- one `h1i` file with all the daily instantaneous output for the month
- thirty-one `h2a` files, one for each day of the month, with 3-hourly time-averaged output

Basically, your goal is to produce:

- one `h1i` file with 31 time samples
- thirty-one `h2a` files with 8 time samples each

Set the run length to 1 month and make a 1-month run.

</div>



<div class="alert alert-warning">  
<details>

<summary> <font face="Times New Roman" color='blue'>Click here for hints</font> </summary>
    
**# How do I compile?**

You can compile with the command:
```
qcmd -- ./case.build
```

**# How do I control the output?**

Use CAM namelist variables:

```fortran
nhtfrq
mfilt
fincl
```

Use the CAM interpolation settings to interpolate history output to a regular 2° × 2° latitude–longitude grid.

Look at the online documentation for these variables.

</details>
</div>




<div class="alert alert-success">   
<details>
<summary><font face="Times New Roman" color='blue'>Click here for the solution</font></summary><br>
                
**# Create a new case**

Create a new case `b1850_high_freq` with the command:

```bash
cd /glade/u/home/$USER/code/my_cesm_code/cime/scripts/

./create_newcase --case ~/cases/b1850_high_freq \
  --compset B1850C_LTso \
  --res ne16pg3_t201 \
  --run-unsupported
```

**# Setup**
    
Invoke `case.setup` with the command:

```bash
cd ~/cases/b1850_high_freq
./case.setup
```

**# Customize namelists**
    
Edit the file `user_nl_cam` and add the lines:

```fortran
 nhtfrq = 0, -24, -3
 mfilt = 1, 31, 8
 fincl2 = 'T:I','Q:I','U:I','V:I'
 fincl3 = 'T','Q','U','V'

 interpolate_output = .true.,.true.,.true.
 interpolate_nlat = 91,91,91
 interpolate_nlon = 180,180,180
```

Note: For `fincl3`, you could write:

```fortran
fincl3 = 'T:A','Q:A','U:A','V:A'
```

But since averaging `:A` is the default for these variables, it is not necessary to specify it explicitly.

 
**# Set run length**
    
Change the run length:

```bash
./xmlchange STOP_N=1,STOP_OPTION=nmonths
```


**# Change the job queue and account number**

If needed, change `job queue` and `account number`. <br>
For instance, to run in the queue `tutorial` and the project number ``UESM0016`` (You should use the project number given for this tutorial), use the command:
```  
./xmlchange JOB_QUEUE=tutorial,PROJECT=UESM0016 --force
```

**# Build and submit**
    
Build the model and submit your job:
```
qcmd -- ./case.build
./case.submit
```

____
    
**# Look at your solution**

When the run is complete, go to the archive directory:

```bash
cd /glade/derecho/scratch/$USER/archive/b1850_high_freq/atm/hist
ls
```

In CESM3, CAM history files are separated into averaged and instantaneous files using the suffixes `a` and `i`.

You should see files like:

```text
- h0 
b1850_high_freq.cam.h0a.0001-01.nc
b1850_high_freq.cam.h0i.0001-02-01-00000.nc

- h1
b1850_high_freq.cam.h1i.0001-01-02-00000.nc

- h2
b1850_high_freq.cam.h2a.0001-01-01-10800.nc
...
b1850_high_freq.cam.h2a.0001-01-31-10800.nc

b1850_high_freq.cam.h2i.0001-01-01-10800.nc
...
b1850_high_freq.cam.h2i.0001-01-31-10800.nc
```

The important files for this exercise are:

* `h1i`: daily instantaneous output requested with `fincl2 = 'T:I','Q:I','U:I','V:I'`
* `h2a`: 3-hourly averaged output requested with `fincl3 = 'T:A','Q:A','U:A','V:A'`

You will also see companion instantaneous or averaged files, such as `h0i` or `h2i`. This is expected in CESM3 because CAM history streams are split into `a` and `i` files.

For this exercise, focus on checking:

```bash
ncdump -h b1850_high_freq.cam.h1i.0001-01-02-00000.nc
ncdump -h b1850_high_freq.cam.h2a.0001-01-01-10800.nc
```

The `h1i` file should contain the instantaneous fields. These variables should have:

```text
cell_methods = "time: point" ;
```

The `h2a` files should contain the time-averaged fields. These variables should include:

```text
cell_methods = "time: mean"
```

Check the number of time samples:

```bash
ncdump -h b1850_high_freq.cam.h1i.0001-01-02-00000.nc | grep time
ncdump -h b1850_high_freq.cam.h2a.0001-01-01-10800.nc | grep time
```

The `h1i` file should contain 31 time samples.

Each `h2a` file should contain 8 time samples.

You can also check file sizes with:

```bash
du -skh /glade/derecho/scratch/$USER/archive/b1850_high_freq/atm/hist/*
```
____
    
**# Why do some filenames end with `00000` while others end with `10800`?**

The last field in a CAM history filename is the number of **seconds since midnight** associated with the output time.

For example:

* `00000` corresponds to **00:00:00 UTC (midnight)**.
* `10800` corresponds to **03:00:00 UTC** (3 × 3600 seconds = 3 hours).

For this exercise:

* The `h1i` stream contains one instantaneous sample per day. The sample is written at the beginning of the day, so the filename ends with `00000`.

* The `h2a` stream contains 3-hourly **time-averaged** output. The first average represents the interval from **00:00 to 03:00**, so the timestamp corresponds to the **end of the averaging period**, giving `10800` (03:00 UTC).

Similarly, the remaining time samples in each `h2a` file correspond to averages ending at 06:00, 09:00, ..., up to 24:00.

Notice that there is also an `h2i` file. It contains the corresponding instantaneous output at the same sampling frequency, with timestamps matching the same output times.




</details>
</div>

